# People Like Me

Setup:  create a .venv -- must be in a workspace to do this.  This .venv will be available to all projects in workspace
Use:  https://chatgpt.com/share/69cd1090-2a98-8328-9f28-b24fa7b362ce

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

# print("Path to dataset files:", path)

#### For HuggingFace, install library "datasets"

In [1]:
# From Hugging Face:
# pip install datasets
# Dataset is located here:  C:\Users\DeloresMincarelli\.cache\huggingface\hub\

import pandas as pd
from datasets import load_dataset

dataset = load_dataset("med2425/resume-job-fit-merged-v1")

In [2]:
import json

row = dataset["train"][0]
print(json.dumps(row, indent=2, ensure_ascii=False))

{
  "resume": "SummarySelf-motivated accountant  offering a strong work ethic and determination to complete tasks in a timely manner. Accurate and detail-oriented with extensive auditing and finance knowledge.\nHighlightsComplex problem solvingStrong communication skillsExpert in customer relationsPortfolio managementAProficient in Microsoft OfficeMicrosoft Excel expertRisk management expertiseFinancial statement analysisGeneral ledger accounting\nExperienceCurrentto08/2014AccountantApartment Income Reit Corp.–Nashua,NH,Prepare unpaid reports on actual expenses for marketing line of business.Create and maintain pending and process able database.Prepare and setup vendor purchase orders contracts as well as CRX templates.Verify funding and SAP project code against the most recent budget/forecast submission.Key invoices into ePurchase system as well as approve and reconcile invoices.Track invoices from submission to payment on database.Monitor invoice central mailbox that will include inv

In [3]:

train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()


train.head()
# row = train.iloc[0].to_dict()
# print(json.dumps(row, indent=2, ensure_ascii=False))

,resume,jd,label,source,resume_domain,jd_domain
0,SummarySelf-motivated accountant offering a s...,"Our client, a Raleigh-based full-service comme...",Potential Fit,generated_smart,finance,finance
1,Professional ProfileSpecialized knowledge of r...,"""All candidates must be directly contracted by...",No Fit,generated_smart,finance,software
2,"SummaryEngineering Project Manager\nAmbitious,...",Calling all innovators find your future at Fi...,No Fit,generated_smart,software,software
3,SummaryData Protection Consultant with 10 year...,"MUST WORK ON A W2 INDEPENDENTLY, NO SPONSORSHI...",No Fit,generated_smart,software,data
4,Career OverviewHighly skilled SOFTWARE QUALITY...,Calling all data nerds who love finance! \nIf ...,No Fit,generated_smart,software,data


In [4]:
distinct_resume_domains = sorted(train["resume_domain"].dropna().unique())
distinct_resume_domains

['ai',
 'data',
 'design',
 'engineering',
 'finance',
 'healthcare',
 'hr',
 'legal',
 'management',
 'marketing',
 'sales',
 'software']

In [5]:
# Cleaning

train_resume = train[["resume", "resume_domain"]].copy()
test_resume = test[["resume", "resume_domain"]].copy()

train_resume = train_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)
test_resume = test_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)

# One ID per distinct resume text (shared across train/test)
combined_resume = pd.concat([train_resume["resume"], test_resume["resume"]], ignore_index=True)
resume_ids, _ = pd.factorize(combined_resume)
n_train = len(train_resume)
train_resume["resume_id"] = resume_ids[:n_train]
test_resume["resume_id"] = resume_ids[n_train:]

train_resume["source"] = "huggingface"
test_resume["source"] = "huggingface"

train_resume["Is_Me"] = 0
test_resume["Is_Me"] = 0

train_resume = train_resume[["resume_id", "resume", "resume_domain", "source", "Is_Me"]]
test_resume = test_resume[["resume_id", "resume", "resume_domain", "source", "Is_Me"]]

train_resume.head()

,resume_id,resume,resume_domain,source,Is_Me
0,0,SummarySelf-motivated accountant offering a s...,finance,huggingface,0
1,1,Professional ProfileSpecialized knowledge of r...,finance,huggingface,0
2,2,"SummaryEngineering Project Manager\nAmbitious,...",software,huggingface,0
3,3,SummaryData Protection Consultant with 10 year...,software,huggingface,0
4,4,Career OverviewHighly skilled SOFTWARE QUALITY...,software,huggingface,0


### Add your resume from a PDF

Install once: `pip install pypdf ipywidgets`

1. Run the next cell (extract helpers).
2. Run the cell after that (optional path + upload widget). If you use **Upload**, pick your PDF, then run the **following** cell again so the widget value is read.
3. Run the last cell to extract text, save it beside the notebook as `my_resume_extracted.txt`, and append one row to `train` (`resume_domain='data'`). Then **re-run** the `train_resume` build cell so your resume is included there too.

In [6]:
import io
import re
from pathlib import Path

from pypdf import PdfReader


def organize_resume_text(text: str) -> str:
    """Normalize whitespace and page breaks for cleaner plain text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_pdf(data: bytes) -> str:
    reader = PdfReader(io.BytesIO(data))
    parts = []
    for page in reader.pages:
        t = page.extract_text()
        if t:
            parts.append(t.strip())
    return organize_resume_text("\n\n".join(parts))


print("Ready: organize_resume_text, extract_text_from_pdf")

Ready: organize_resume_text, extract_text_from_pdf


### Use Upload button to upload pdf resume

In [7]:
import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
# PDF in the same folder as the notebook (when Jupyter cwd is this folder)
MY_RESUME_PDF = NOTEBOOK_DIR / "dam_resume.pdf"

print(f"If no upload button appears (common in Cursor), use a PDF at:\n  {MY_RESUME_PDF.resolve()}\n")

pdf_upload = widgets.FileUpload(accept=".pdf", multiple=False, description="Resume PDF")
display(pdf_upload)

If no upload button appears (common in Cursor), use a PDF at:
  C:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\dam_resume.pdf



FileUpload(value=(), accept='.pdf', description='Resume PDF')

### Scrape pdf text and save as my_resume_extract.txt

In [8]:

RESUME_TEXT_CACHE = NOTEBOOK_DIR / "my_resume_extracted.txt"


def _pdf_bytes_from_upload(w) -> bytes | None:
    if not w.value:
        return None
    f0 = w.value[0]
    content = f0["content"]
    if hasattr(content, "tobytes"):
        return content.tobytes()
    return bytes(memoryview(content))


pdf_bytes = _pdf_bytes_from_upload(pdf_upload)
if pdf_bytes is not None:
    print("Using PDF from upload widget.")
    my_resume_text = extract_text_from_pdf(pdf_bytes)
    RESUME_TEXT_CACHE.write_text(my_resume_text, encoding="utf-8")
elif MY_RESUME_PDF.exists():
    pdf_bytes = MY_RESUME_PDF.read_bytes()
    print(f"Using PDF from disk: {MY_RESUME_PDF.resolve()}")
    my_resume_text = extract_text_from_pdf(pdf_bytes)
    RESUME_TEXT_CACHE.write_text(my_resume_text, encoding="utf-8")
elif RESUME_TEXT_CACHE.exists():
    my_resume_text = RESUME_TEXT_CACHE.read_text(encoding="utf-8")
    print(f"Using saved text from: {RESUME_TEXT_CACHE.resolve()}")
else:
    raise FileNotFoundError(
        f"No resume source yet: use the upload widget or save a PDF at:\n  {MY_RESUME_PDF.resolve()}\n"
        f"Or create: {RESUME_TEXT_CACHE.resolve()}"
    )

# Idempotent: drop prior personal rows so re-running this cell does not duplicate
train = train[train["source"] != "me"].reset_index(drop=True)
test = test[test["source"] != "me"].reset_index(drop=True)

new_row = pd.DataFrame(
    [
        {
            "resume": my_resume_text,
            "jd": "",
            "label": pd.NA,
            "source": "me",
            "resume_domain": "data",
            "jd_domain": pd.NA,
            "Is_Me": 1,
        }
    ]
)
train = pd.concat([train, new_row], ignore_index=True)
test = pd.concat([test, new_row], ignore_index=True)

print(
    f"Resume text: {len(my_resume_text):,} chars. "
    f"train rows: {len(train)}, test rows: {len(test)}"
)

Using PDF from upload widget.
Saved text to c:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\my_resume_extracted.txt (5,427 chars). train rows: 80018
